In [1]:
##### Bibliotecas
import os
import torch
import numpy as np
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import datasets, models, transforms
import matplotlib.pyplot as plt

# Ignite
from ignite.engine import Engine, Events
from ignite.handlers import EarlyStopping
from ignite.metrics import Accuracy, Loss

# Optuna
import optuna

# Organização do dataset
data = "/home/jovyan/DADOS-DIVIDIDOS"
feature_extract = True

In [2]:
data_transforms = {
    'train': transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(30),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.2),
        transforms.RandomVerticalFlip(),
        transforms.RandomPerspective(distortion_scale=0.2, p=0.5, interpolation=3, fill=0),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize(224),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'test': transforms.Compose([
        transforms.Resize(224),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
}

image_datasets = {x: datasets.ImageFolder(os.path.join(data, x), data_transforms[x]) for x in ['train', 'val', 'test']}

/opt/conda/lib/python3.10/site-packages/torchvision/transforms/transforms.py:768: UserWarning: Argument 'interpolation' of type int is deprecated since 0.13 and will be removed in 0.15. Please use InterpolationMode enum.
  warnings.warn(


In [3]:
# Extração de features + Congelamento dos parâmetros
def set_parameter_requires_grad(model, feature_extracting):
    if feature_extracting:
        for param in model.parameters():
            param.requires_grad = False
            
            

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.mobilenet_v3_large(pretrained=False)
set_parameter_requires_grad(model, feature_extract)

model_mobile_net = "/home/jovyan/models/mobilenet_v3_large-model-84.pth"
state_dict = torch.load(model_mobile_net)

del state_dict['classifier.3.weight']
del state_dict['classifier.3.bias']

model.load_state_dict(state_dict, strict=False)
num_features = model.classifier[0].in_features
model.to(device)


/opt/conda/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


MobileNetV3(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
      (2): Hardswish()
    )
    (1): InvertedResidual(
      (block): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=16, bias=False)
          (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
          (2): ReLU(inplace=True)
        )
        (1): Conv2dNormActivation(
          (0): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
        )
      )
    )
    (2): InvertedResidual(
      (block): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 64, kernel_size=(1, 1), stride=(1, 1), bi

In [5]:
# Função de treinamento e validação
def train_step(engine, batch):
    x, y = batch
    x, y = x.to(device), y.to(device)

    model.train()
    y_pred = model(x)
    loss = criterion(y_pred, y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    return loss.item()

def validation_step(engine, batch):
    model.eval()
    with torch.no_grad():
        x, y = batch
        x, y = x.to(device), y.to(device)
        y_pred = model(x)
        return y_pred, y

# Função `objective` para busca bayesiana com Optuna
def objective(trial):
    global model, optimizer, criterion
    
    # Hiperparâmetros
    dropout_rate1 = trial.suggest_uniform("dropout1", 0.2, 0.5)
    dropout_rate2 = trial.suggest_uniform("dropout2", 0.2, 0.5)
    num_neurons_fc1 = trial.suggest_categorical("num_neurons_fc1", [256, 512, 1024])
    num_neurons_fc2 = trial.suggest_categorical("num_neurons_fc2", [128, 256, 512])
    activation = trial.suggest_categorical("activation", ["ReLU"])
    batch_size = trial.suggest_categorical("batch_size", [128])
    optimizer_name = trial.suggest_categorical("optimizer", ["SGD", "Adam"])
    lr = trial.suggest_loguniform("lr", 1e-5, 1e-2)
    momentum = trial.suggest_uniform("momentum", 0.7, 0.99) if optimizer_name == "SGD" else None

    # Função de ativação
    activation_fn = getattr(nn, activation)()

    # Redefinir o DataLoader com o batch size sugerido
    dataloaders_dict = {
        'train': torch.utils.data.DataLoader(image_datasets['train'], batch_size=batch_size, shuffle=True),
        'val': torch.utils.data.DataLoader(image_datasets['val'], batch_size=batch_size, shuffle=False)
    }

    # Configurar o modelo com os hiperparâmetros sugeridos
    model.classifier = nn.Sequential(
        nn.Dropout(p=dropout_rate1),
        nn.Linear(num_features, num_neurons_fc1),
        activation_fn,
        nn.Dropout(p=dropout_rate2),
        nn.Linear(num_neurons_fc1, num_neurons_fc2),
        activation_fn,
        nn.Linear(num_neurons_fc2, 2)
    )
    model.to(device)

    # Configurar o otimizador
    params_to_update = [p for p in model.parameters() if p.requires_grad]
    if optimizer_name == "SGD":
        optimizer = optim.SGD(params_to_update, lr=lr, momentum=momentum)
    else:
        optimizer = optim.Adam(params_to_update, lr=lr)

    # Definir função de perda
    criterion = nn.CrossEntropyLoss()

    # Ignite trainers
    trainer = Engine(train_step)
    evaluator = Engine(validation_step)

    # Métricas
    val_metrics = {
        "accuracy": Accuracy(),
        "loss": Loss(criterion)
    }
    for name, metric in val_metrics.items():
        metric.attach(evaluator, name)

    @trainer.on(Events.EPOCH_COMPLETED)
    def log_training_results(engine):
        evaluator.run(dataloaders_dict['val'])
        metrics = evaluator.state.metrics
        print(f"Val Accuracy: {metrics['accuracy']:.4f}")
        
        # Early Stopping
        score_function = lambda engine: engine.state.metrics['accuracy']
        handler = EarlyStopping(patience=10, score_function=score_function, trainer=trainer)
        evaluator.add_event_handler(Events.COMPLETED, handler)

    # Executar o treinamento
    trainer.run(dataloaders_dict['train'], max_epochs=100)

    # Obter a acurácia final
    evaluator.run(dataloaders_dict['val'])
    return evaluator.state.metrics["accuracy"]
# Rodar o estudo Optuna
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

# Melhor conjunto de hiperparâmetros
print("Best trial:")
trial = study.best_trial
print(f"Accuracy: {trial.value}")
print("Best hyperparameters: ", trial.params)

with open("result_mobile_net.txt", 'a', encoding='utf-8') as file:
    file.write(f'Best trial --- \n Accuracy: {trial.value}\n \n \n')
    file.write(f'Best hyperparameters : {trial.params}')

[I 2025-02-06 23:24:06,767] A new study created in memory with name: no-name-9e5455a1-6772-419c-85a4-02ecec9342a9
/tmp/ipykernel_135/3490020165.py:29: FutureWarning: suggest_uniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float instead.
  dropout_rate1 = trial.suggest_uniform("dropout1", 0.2, 0.5)
/tmp/ipykernel_135/3490020165.py:30: FutureWarning: suggest_uniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float instead.
  dropout_rate2 = trial.suggest_uniform("dropout2", 0.2, 0.5)
/tmp/ipykernel_135/3490020165.py:36: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  lr = trial.suggest_loguniform("lr", 1e-5, 1e-2)


Val Accuracy: 0.9267
Val Accuracy: 0.9780
Val Accuracy: 0.9817
Val Accuracy: 0.9817
Val Accuracy: 0.9707
Val Accuracy: 0.9744
Val Accuracy: 0.9780
Val Accuracy: 0.9670
Val Accuracy: 0.9744
Val Accuracy: 0.9744
Val Accuracy: 0.9890
Val Accuracy: 0.9853
Val Accuracy: 0.9890
Val Accuracy: 0.9707
Val Accuracy: 0.9890
Val Accuracy: 0.9853
Val Accuracy: 0.9853
Val Accuracy: 0.9963
Val Accuracy: 0.9890
Val Accuracy: 0.9890
Val Accuracy: 0.9780
Val Accuracy: 0.9780
Val Accuracy: 0.9780
Val Accuracy: 0.9817
Val Accuracy: 0.9927
Val Accuracy: 0.9817
Val Accuracy: 0.9890


2025-02-07 00:40:00,479 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-07 00:40:00,479 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-07 00:40:00,480 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-07 00:40:00,480 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-07 00:40:00,480 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-07 00:40:00,481 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-07 00:40:00,481 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-07 00:40:00,481 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-07 00:40:00,481 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-07 00:40:00,482 ignite.handlers.early_stopping.EarlyStop

Val Accuracy: 0.9890


2025-02-07 00:40:28,673 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-07 00:40:28,674 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-07 00:40:28,674 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-07 00:40:28,675 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-07 00:40:28,675 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-07 00:40:28,675 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-07 00:40:28,675 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-07 00:40:28,676 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-07 00:40:28,676 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-07 00:40:28,676 ignite.handlers.early_stopping.EarlyStop

...

Val Accuracy: 0.9158
Val Accuracy: 0.9414
Val Accuracy: 0.9524
Val Accuracy: 0.9487
Val Accuracy: 0.9597
Val Accuracy: 0.9634
Val Accuracy: 0.9634
Val Accuracy: 0.9853
Val Accuracy: 0.9853
Val Accuracy: 0.9744
Val Accuracy: 0.9780
Val Accuracy: 0.9853
Val Accuracy: 0.9853
Val Accuracy: 0.9780
Val Accuracy: 0.9927
Val Accuracy: 0.9744
Val Accuracy: 0.9853
Val Accuracy: 0.9853
Val Accuracy: 0.9890
Val Accuracy: 0.9780
Val Accuracy: 0.9853
Val Accuracy: 0.9817
Val Accuracy: 0.9780
Val Accuracy: 0.9817


2025-02-10 02:23:32,080 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-10 02:23:32,081 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-10 02:23:32,081 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-10 02:23:32,081 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-10 02:23:32,081 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-10 02:23:32,082 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-10 02:23:32,082 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-10 02:23:32,082 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-10 02:23:32,082 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-10 02:23:32,083 ignite.handlers.early_stopping.EarlyStop

Val Accuracy: 0.9890


2025-02-10 02:24:00,538 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-10 02:24:00,539 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-10 02:24:00,539 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-10 02:24:00,539 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-10 02:24:00,540 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-10 02:24:00,540 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-10 02:24:00,541 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-10 02:24:00,541 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-10 02:24:00,542 ignite.handlers.early_stopping.EarlyStopping INFO: EarlyStopping: Stop training
2025-02-10 02:24:00,543 ignite.handlers.early_stopping.EarlyStop

Best trial:
Accuracy: 0.9963369963369964
Best hyperparameters:  {'dropout1': 0.21269401273868907, 'dropout2': 0.3333736211255451, 'num_neurons_fc1': 512, 'num_neurons_fc2': 512, 'activation': 'ReLU', 'batch_size': 128, 'optimizer': 'SGD', 'lr': 0.0019134979136296384, 'momentum': 0.9899652823553813}
